In [16]:
from dotenv import load_dotenv
load_dotenv()  # This looks for your .env file and loads the variables

from openai import OpenAI
client = OpenAI() # Now it will find your API key perfectly!

In [17]:
import openai  # or your preferred LLM client
from gitsource import GithubRepositoryDataReader

# 1. Initialize the reader with the specific course commit
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

# 2. Download the files
print("Fetching files from GitHub...")
files = reader.read()

# 3. Parse files into your documents list
documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

# 4. Check the total count (This answers Q1!)
print(f"Total lesson pages found: {len(documents)}")

Fetching files from GitHub...
Total lesson pages found: 72


In [18]:
import minsearch

# 1. Initialize the index specifying text and keyword fields
index = minsearch.Index(
    text_fields=["content"],      # The field we perform the full-text search on
    keyword_fields=["filename"]   # The field used for precise filtering/grouping
)

# 2. Fit the index with your list of parsed documents
index.fit(documents)

# 3. Define your search query
query = "How does the agentic loop keep calling the model until it stops?"

# 4. Perform the search
# We don't have explicit filter or boost requirements for Q2, so we just pass the query.
results = index.search(query=query, num_results=5)

# 5. Get the filename of the first result
if results:
    print("First result filename:", results[0]['filename'])
else:
    print("No results found.")
    

First result filename: 01-agentic-rag/lessons/14-agentic-loop.md


In [19]:
import tiktoken

# Get the tokenizer for the model family
encoding = tiktoken.encoding_for_model("gpt-4o") # or similar mini models

# Estimate the tokens in your prompt string
token_count = len(encoding.encode(prompt))
print(f"Estimated Input Tokens: {token_count}")

Estimated Input Tokens: 7118


In [20]:
from openai import OpenAI

# Initialize the OpenAI client (using your specified provider/model configuration)
client = OpenAI()

# 1. Search the index built from Q2 to fetch context documents
query = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(query=query, num_results=5)

# 2. Build the context using 'content' instead of the FAQ schema fields
context_entries = []
for doc in search_results:
    context_entries.append(f"Filename: {doc['filename']}\nContent:\n{doc['content']}")

context = "\n\n".join(context_entries)

# 3. Define the prompt template
prompt_template = """
You're a course teaching assistant. Answer the user QUESTION based on the CONTEXT provided.
Use only the facts from the CONTEXT. If the context doesn't contain the answer, say that you don't know.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

prompt = prompt_template.format(question=query, context=context)

# 4. Invoke the model and track token usage
response = client.chat.completions.create(
    model="gpt-5.4-mini", # or your configured model
    messages=[{"role": "user", "content": prompt}]
)

# 5. Extract and print the input token counts
input_tokens = response.usage.prompt_tokens
print(f"Input (Prompt) Tokens: {input_tokens}")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
from gitsource import chunk_documents

# Chunk the original documents using a sliding window
chunks = chunk_documents(documents, size=2000, step=1000)

# Check the total number of chunks generated
print(f"Total number of chunks: {len(chunks)}")

In [ ]:
import minsearch
import tiktoken

# 1. Initialize and fit a new index on the CHUNKS
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

# 2. Search using the same query
query = "How does the agentic loop keep calling the model until it stops?"
chunk_results = chunk_index.search(query=query, num_results=5)

# 3. Build the chunked context
chunk_context_entries = []
for chunk in chunk_results:
    chunk_context_entries.append(f"Filename: {chunk['filename']}\nContent:\n{chunk['content']}")

chunk_context = "\n\n".join(chunk_context_entries)

# 4. Reconstruct the prompt template
prompt_template = """
You're a course teaching assistant. Answer the user QUESTION based on the CONTEXT provided.
Use only the facts from the CONTEXT. If the context doesn't contain the answer, say that you don't know.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

chunk_prompt = prompt_template.format(question=query, context=chunk_context)

# 5. Calculate the tokens using the token counter
encoding = tiktoken.encoding_for_model("gpt-4o")
chunk_tokens = len(encoding.encode(chunk_prompt))

print(f"Chunked version input tokens: {chunk_tokens}")